#15.773 Homework 2 (Spring 2024): Natural Langugage Processing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)

# Introduction

This homework assignment will ask you to build text classification models using BoW, GloVE and BERT as well as compare the performance of the three models.

We will work with a famous dataset in natural langauge processing called **20 Newsgroup**,  which consists of posts from an online forum under topics such as politics, religion, sports...etc. As the name suggests, there are a total of 20 topics in this dataset. The 20 Newsgroup dataset is a popular benchmark for text classification algorithms.

The entire dataset is quite large. To ensure training takes a reasonable amount of time, we will only choose 6 out of the 20 topics, including topics like religion, space (astronomy) and medicine.

In [2]:
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(subset='train', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'])
newsgroups_test = fetch_20newsgroups(subset='test', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'])

train_df = pd.DataFrame({'text': newsgroups_train.data, 'label': newsgroups_train.target})
test_df = pd.DataFrame({'text': newsgroups_test.data, 'label': newsgroups_test.target})

print(f"""
Train samples: {train_df.shape[0]}
Test samples: {test_df.shape[0]}
""")

train_df.head(15)


Train samples: 3222
Test samples: 2145



,text,label
0,From: boylan@pi.eai.iastate.edu (Terran Boylan...,1
1,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,0
2,From: snichols@adobe.com (Sherri Nichols)\nSub...,3
3,From: johnm@spudge.lonestar.org (John Munsch)\...,1
4,From: Nanci Ann Miller <nm0w+@andrew.cmu.edu>\...,0
5,From: nsmca@aurora.alaska.edu\nSubject: 30826\...,4
6,From: geb@cs.pitt.edu (Gordon Banks)\nSubject:...,3
7,From: higgins@fnalf.fnal.gov (Bill Higgins-- B...,4
8,From: mmm@cup.portal.com (Mark Robert Thorson)...,3
9,From: daniel@lclark.edu (Daniel Snodgrass)\nSu...,1


The distribution of labels across these 6 classes is fairly balanced.



In [3]:
train_df['label'].value_counts() / train_df.shape[0]

,count
label,
2,0.184358
3,0.184358
4,0.184047
1,0.181254
0,0.148976
5,0.117008


Let's convert our dependent variable into a 1-hot-encoded vector.

In [4]:
# Let's turn the target into a dummy vector
y_train = pd.get_dummies(train_df['label']).to_numpy(dtype='int64')
y_test = pd.get_dummies(test_df['label']).to_numpy(dtype='int64')

y_train[:10]

array([[0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0]])

# Problem 1: Bag-of-Words (BoW) Model [30 Points]

In this problem, we will build a bag-of-words model using the text vectorization capabilities of Keras. We will then change some of the parameters of this vectorization process and see how it changes the performance.



## Part (a): Build a Base Model [15 Points]

**Text Vectorization**

Please fill in the code in the following cell. We would like to create a text vectorization layer which uses:

* Maximum of 2000 tokens.
* Unigrams
* Outputs a multi-hot encoded BoW encoding
* Converts text to lower case and strips punctuation

In [5]:
# Set the maximum number of tokens (which is the size of the vocabulary) to 1000
max_tokens = 2000

# Configure the text vectorization layer
text_vectorization = keras.layers.TextVectorization(output_mode='multi_hot',max_tokens=max_tokens,standardize='lower_and_strip_punctuation', split='whitespace',ngrams=1)

# Let's adapt the Text Vectorization layer using the training corpus
text_vectorization.adapt(train_df['text'])

# We vectorize our input with the adapted Text Vectorization layer
X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

pd.DataFrame(X_train, columns = text_vectorization.get_vocabulary())

,[UNK],the,of,to,a,and,in,is,i,that,...,loving,learning,jaegerbuphybuedu,hour,helps,glutamate,finding,delta,careful,bother
0,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1,1,1,1,...,0,0,1,0,0,0,0,0,0,0
2,1,1,0,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3218,1,1,1,1,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3219,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3220,1,1,1,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In the following cell, please build a simple Neural Network with a single hidden layer of 128 neurons and a ReLu activation function. Be sure to specify the shape of the input correctly and use the appropriate activation function for the output. Your model should have 256902 parameters. The code for compiling, training and computing the test accuracy has been written for you already.

In [22]:
### YOUR CODE BELOW ###
inputs = keras.layers.Input(shape=(max_tokens,))

x = keras.layers.Dense(128, activation='relu')(inputs)
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       256,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,902 (1003.52 KB)

 Trainable params: 256,902 (1003.52 KB)

 Non-trainable params: 0 (0.00 B)

A theme that we will be investigating in this homework is the **impact of the number of training examples on the various models**. As a result, in addition to training the model on all 3000+ training examples, we will also test how our model performs with only 200 examples.

The code below shows a fitting process in which we first fit model `bow_model` to the first 200 examples from the training set `(X_train[:200], y_train[:200]`). We print the accuracy of the model on the test set. Then, we run the training procedure for another 20 epochs, this time with the full data set `(X_train, y_train`).

In [23]:
bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.3350 - loss: 1.7121
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9150 - loss: 1.1147 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9800 - loss: 0.7378 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9900 - loss: 0.4665 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2910 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1827 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1176 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0789 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0556 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0411 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0317 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0253 
E

Please write down the test accuracy shown above and comment on the model's performance. What is the baseline performance by predicting the most frequent class?

**<font color='red'>Answer:</font>**

BoW Base Model
* Accuracy with 200 Examples: 63%.0%
* Accuracy with All Examples: 83.0%

Baseline in this case is around 17%. With fewer examples we achieve accuracy of about 63%. This is obviously corrected by using more examples and took it huge jump to 80%.

## Part (b): Exploring Hyperparameters [15 Points]

Now, let us try changing some of the hyperparameters that in the text vectorization process.

For each of the following modifications, please change the code for Problem 1(a), train the model and report the out-of-sample accuracy below. You do not need to duplicate the code from 1(a) here.

* Use bigrams instead of unigrams.
* Increase the maximum number of tokens from 2000 to 5000
* Use count vectorization instead of multi-hot.

Note that you should make each change independent of the other two changes. <font color='red'>Don't forget to undo each change before making the next </font>

Copy and paste the code from Problem 1(a) into a single cell. Then, make each of the 3 changes above and fill in the table of accuracies.




In [24]:
max_tokens = 2000

text_vectorization = keras.layers.TextVectorization(output_mode='multi_hot',
                                                    max_tokens=max_tokens,
                                                    standardize='lower_and_strip_punctuation',
                                                    split='whitespace',ngrams=2
)

text_vectorization.adapt(train_df['text'])

X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

### YOUR CODE BELOW ###
inputs = keras.Input(shape=(max_tokens,))
x = keras.layers.Dense(128, activation='relu')(inputs)

outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       256,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,902 (1003.52 KB)

 Trainable params: 256,902 (1003.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - accuracy: 0.2300 - loss: 1.7806
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8700 - loss: 1.0853 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9900 - loss: 0.6906 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.4275 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.2626 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.1636 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1053 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0711 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0506 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0379 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0296 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0239 
E

In [25]:
max_tokens = 5000

text_vectorization = keras.layers.TextVectorization(output_mode='multi_hot',
                                                    max_tokens=max_tokens,
                                                    standardize='lower_and_strip_punctuation',
                                                    split='whitespace',ngrams=1
)

text_vectorization.adapt(train_df['text'])

X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

### YOUR CODE BELOW ###
inputs = keras.Input(shape=(max_tokens,))
x = keras.layers.Dense(128, activation='relu')(inputs)

outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 640,902 (2.44 MB)

 Trainable params: 640,902 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 0.3150 - loss: 1.7298
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9900 - loss: 0.9621 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.5296 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2725 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.1412 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0780 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0469 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0309 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0221 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0168 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0134 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0111 
E

In [26]:
max_tokens = 2000

text_vectorization = keras.layers.TextVectorization(output_mode='count',
                                                    max_tokens=max_tokens,
                                                    standardize='lower_and_strip_punctuation',
                                                    split='whitespace',ngrams=1
)

text_vectorization.adapt(train_df['text'])

X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

### YOUR CODE BELOW ###
inputs = keras.Input(shape=(max_tokens,))
x = keras.layers.Dense(128, activation='relu')(inputs)

outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       256,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,902 (1003.52 KB)

 Trainable params: 256,902 (1003.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - accuracy: 0.2400 - loss: 2.2523
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5700 - loss: 1.3254 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7600 - loss: 0.9235 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9650 - loss: 0.5707 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9900 - loss: 0.4311 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3078 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2287 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1725 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1323 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.1032 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0819 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0657 
E

Replace the following numbers with the out-of-sample accuracy you obtained from re-running Problem 1(a)'s code with the relevant modifications.

For each of the 3 modifications, discuss why one might expect such a change to be beneficial to the model's out-of-sample accuracy.

Using the actual accuracies, comment on whether each modification improved, hurt or didn't affect accuracy from the baseline model in (a).

**<font color='red'>Answer:</font>**

* Bigrams instead of unigrams
  * 200 Training Examples: 55.5%
  * All Training Examples: 77.5%
  * **Discussion**: Adding `ngrams=2` i.e. bigrams doesn't really increases the accuracy rather it decreases it. Possible reason behind this is that because we are not increassing the `max_tokens` size although we are doing word pairing so many possible context is lost because of unnecessary pairs like e.g. 'on the' which do not contribute for the training of model.
* Increasing max_tokens to 5000.
  * 200 Training Examples: 66.6%
  * All Training Examples: 86.9%
  * **Discussion**: Accuracy is incresed 4% because of increase in tokens since context was added which was left out due to some tokens which were exceeding the limit in earlier model.
* Count instead of multi-hot
  * 200 Training Examples: 62.4%
  * All Training Examples: 83.1%
  * **Discussion**: This almost stays the same. Possible reason behind this is `count` will affect if frequency of word affects class. E.g. Sentiment Analysis, whichever type i.e. +ve or -ve word affects the sentiment of sentence.  

# Problem 2: GloVe without Fine-Tuning [20 Points]

Now, we will train a model using GloVe. We will try using GloVe vectors out of box and with fine-tuning. Let's first download the GloVe vectors from the internet.

In [27]:
!wget -O glove.6B.300d.zip https://www.dropbox.com/scl/fi/ree568j1h690ktpcn729b/glove.6B.300d.zip?rlkey=eq9r67js0n0vm2znvdev9rerf&dl=1
!unzip glove.6B.300d.zip

--2026-10-11 06:10:29--  https://www.dropbox.com/scl/fi/ree568j1h690ktpcn729b/glove.6B.300d.zip?rlkey=eq9r67js0n0vm2znvdev9rerf
Resolving www.dropbox.com (www.dropbox.com)... 162.125.81.18, 2620:100:6031:18::a27d:5112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.81.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc959e4681a98203657968cd4421.dl.dropboxusercontent.com/cd/0/inline/DJyJZAnHNPN_asZeiFOIoTLGKlq3vSeGO3Cj9OBhuj7bB3VynAx1zjhBID7kT1UWtJXO-P9vuQ3VDOucspadnFUkh_zHd96GsQNXinT0pyGLU96c21y7azE6bvnOjc_aTsrSySAyW-UlwpsNJZ4qKxmN/file# [following]
--2026-10-11 06:10:29--  https://uc959e4681a98203657968cd4421.dl.dropboxusercontent.com/cd/0/inline/DJyJZAnHNPN_asZeiFOIoTLGKlq3vSeGO3Cj9OBhuj7bB3VynAx1zjhBID7kT1UWtJXO-P9vuQ3VDOucspadnFUkh_zHd96GsQNXinT0pyGLU96c21y7azE6bvnOjc_aTsrSySAyW-UlwpsNJZ4qKxmN/file
Resolving uc959e4681a98203657968cd4421.dl.dropboxusercontent.com (uc959e4681a98203657968cd4421.dl.dropboxusercontent.com)... 162.125.81.1

## Part (a): Load Pre-trained Vectors [10 Points]

Let's take the `glove.6B.300d.txt` file, which consists of 300-dimensional vectors for 400,000 words in English.

In [28]:
embedding_dim = 300
path_to_glove_file = f"glove.6B.{embedding_dim}d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


For any of thes 400,000 words, we can query its 300-dimensional vector

In [29]:
embeddings_index['hello']

array([-3.3712e-01, -2.1691e-01, -6.6365e-03, -4.1625e-01, -1.2555e+00,
       -2.8466e-02, -7.2195e-01, -5.2887e-01,  7.2085e-03,  3.1997e-01,
        2.9425e-02, -1.3236e-02,  4.3511e-01,  2.5716e-01,  3.8995e-01,
       -1.1968e-01,  1.5035e-01,  4.4762e-01,  2.8407e-01,  4.9339e-01,
        6.2826e-01,  2.2888e-01, -4.0385e-01,  2.7364e-02,  7.3679e-03,
        1.3995e-01,  2.3346e-01,  6.8122e-02,  4.8422e-01, -1.9578e-02,
       -5.4751e-01, -5.4983e-01, -3.4091e-02,  8.0017e-03, -4.3065e-01,
       -1.8969e-02, -8.5670e-02, -8.1123e-01, -2.1080e-01,  3.7784e-01,
       -3.5046e-01,  1.3684e-01, -5.5661e-01,  1.6835e-01, -2.2952e-01,
       -1.6184e-01,  6.7345e-01, -4.6597e-01, -3.1834e-02, -2.6037e-01,
       -1.7797e-01,  1.9436e-02,  1.0727e-01,  6.6534e-01, -3.4836e-01,
        4.7833e-02,  1.6440e-01,  1.4088e-01,  1.9204e-01, -3.5009e-01,
        2.6236e-01,  1.7626e-01, -3.1367e-01,  1.1709e-01,  2.0378e-01,
        6.1775e-01,  4.9075e-01, -7.5210e-02, -1.1815e-01,  1.86

Let's create our vocabulary space from the training data. We do this using the `TextVectorization` layer just like before. However, this time, we choose `output_mode=int`.

In [30]:
max_length = 400 # For each piece of text, consider only the first 400 tokens (then truncate or pad)
max_tokens = 2000 # Our vocabulary space of tokens can be at most size 5000

text_vectorization_glove = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

text_vectorization_glove.adapt(train_df['text'])

X_train_glove = text_vectorization_glove(train_df['text'])
X_test_glove = text_vectorization_glove(test_df['text'])

X_train_glove

<tf.Tensor: shape=(3222, 400), dtype=int64, numpy=
array([[  14,    1,    1, ...,    0,    0,    0],
       [  14,    1, 1334, ...,  726,    1,    1],
       [  14,    1,    1, ...,    0,    0,    0],
       ...,
       [  14,    1,   27, ...,  116,    1,    6],
       [  14,  546,  325, ...,    0,    0,    0],
       [  14,    1,    1, ...,    0,    0,    0]])>

In the output above for `X_train_glove`, each row is a training example and the columns represent the _index_ of the token. For example, the first training example starts with token 14, followed by token 1, then token 1,... and ending with the padding tokens 0.

**Question**: What English word does token 14 correspond to? Hint: A cell from the Introduction will help you here.

<font color='red'>**Answer:**</font>

Every one of sentences above are starting as `From: someone@gmail.com`. So, here you can conclude that 14 represent the word `From` since each vector in `X_train_glove` is also starting with 14.

For each of the 2000 tokens we extracted from our training data, we will look up its vector from `embedding_vector`.

In [31]:
vocabulary = text_vectorization_glove.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

counter = 0
embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    else:
        counter += 1

embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.04656   ,  0.21318001, -0.0074364 , ...,  0.0090611 ,
        -0.20988999,  0.053913  ],
       ...,
       [-0.038683  ,  0.32398999,  0.10977   , ...,  0.10819   ,
        -0.22115   , -0.17591   ],
       [ 0.47946   ,  0.23098999, -0.11539   , ..., -0.26082999,
        -0.30032   ,  0.089238  ],
       [ 0.19551   , -0.24070001,  0.36401001, ..., -0.069489  ,
         0.32620001,  0.099097  ]])

The above matrix is of shape 2000 by 300. Each row corresponds to one of the 2000 tokens and the value in the row is the 300-dimensional vectorization of that token. We notice that the first two tokens above (index 0 and index 1) are mapped to the all zeros vector.

**Question:** What tokens are represented by index 0 and index 1?

<font color='red'>**Answer:**</font>

Index 0 = `[PAD]`

Index 1 = `[UNK]`

## Part (b): Build the Model [10 Points]
Now, let's build the model. Fill in the missing code below.

* Create an embedding layer of the appropriate input and output dimensions. Initialize it to `keras.initializers.Constant(embedding_matrix)`, set `trainable=False` and `mask_zero=True`.
* Add a `GlobalAveragePooling1D` layer.
* Followed by a Dense layer with 128 neurons (ReLU activation), then a Dropout layer with probability of dropout 0.2. Follow this by another Dense layer of 128 neurons and a Dropout with 0.2 probability.


Your model should have 655,814 parameters out of which 55,814 are trainable.

In [32]:
embedding_layer = keras.layers.Embedding(max_tokens, embedding_dim,embeddings_initializer=keras.initializers.Constant(embedding_matrix), trainable=False, mask_zero=True)

### YOUR CODE BELOW ###
inputs = keras.Input(shape=(max_length,))
x = embedding_layer(inputs)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

glove_model = keras.Model(inputs, outputs)
glove_model.summary()

# Compile model using Adam
glove_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 10 epochs and batch size of 32
glove_model.fit(
    x=X_train_glove[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples: %.3f ***\n" % glove_model.evaluate(x=X_test_glove, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
glove_model.fit(
    x=X_train_glove, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples: %.3f ***\n" % glove_model.evaluate(x=X_test_glove, y=y_test)[1])

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 400)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 400, 300)  │    600,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 400)       │          0 │ input_layer_6[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ embedding_1[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │     38,528 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 128)       │     16,512 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 6)         │        774 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 655,814 (2.50 MB)

 Trainable params: 55,814 (218.02 KB)

 Non-trainable params: 600,000 (2.29 MB)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 277ms/step - accuracy: 0.2100 - loss: 1.7878
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2350 - loss: 1.7625 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2750 - loss: 1.7431 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3100 - loss: 1.7192 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3400 - loss: 1.7196 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3250 - loss: 1.6905 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3800 - loss: 1.6598 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3800 - loss: 1.6199 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4400 - loss: 1.5804 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4250 - loss: 1.5437 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5050 - loss: 1.4974 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5250 - loss: 1.4436  

Report the performance of our Glove Model above which uses the pre-trained word embeddings without fine-tuning. Comment on this performance. Please propose an explanation for why you think GloVe did not perform as well as BoW.

**<font color='red'>Your Answer Here</font>**

* 200 Examples: 63.0%
* All Examples: 80.0%

_Please replace this text with your response_.


# Problem 3: GloVe with Fine Tuning [30 Points]

In problem 2, we did not fine-tune the word embeddings from GloVe. We simply downloaded them from the internet and used them as-is. In this question, we will fine-tune the embeddings to the training data set.

## Part (a): Why Fine Tuning is Needed [10 Points]

The goal of this part is to motivate why fine tuning of the word embeddings is necessary. In the process, we will understand why BoW is able to perform so well over GloVe.

The 20-newsgroup dataset consists of posts from a news discussion forum (from back in the day). Each of these posts look like an email, with a From: someone@email.com and subject lines. This is called the **header** of the post. Similarly, the email also has a **footer**, which typically consists of the author's name and affiliation. Consider one of the training examples shown below:

In [33]:
print(train_df['text'][2101])

From: henry@zoo.toronto.edu (Henry Spencer)
Subject: Re: HLV for Fred (was Re: Prefab Space Station?)
Article-I.D.: zoo.C51875.67p
Organization: U of Toronto Zoology
Lines: 28

In article <C5133A.Gzx@news.cso.uiuc.edu> jbh55289@uxa.cso.uiuc.edu (Josh Hopkins) writes:
>>>Titan IV launches ain't cheap 
>>Granted. But that's because titan IV's are bought by the governemnt. Titan
>>III is actually the cheapest way to put a pound in space of all US expendable
>>launchers.
>
>In that case it's rather ironic that they are doing so poorly on the commercial
>market.  Is there a single Titan III on order?

The problem with Commercial Titan is that MM has made little or no attempt
to market it.  They're basically happy with their government business and
don't want to have to learn how to sell commercially.

A secondary problem is that it is a bit big.  They'd need to go after
multi-satellite launches, a la Ariane, and that complicates the marketing
task quite significantly.

They also had some pr

The header would be

```
From: henry@zoo.toronto.edu (Henry Spencer)
Subject: Re: HLV for Fred (was Re: Prefab Space Station?)
Article-I.D.: zoo.C51875.67p
Organization: U of Toronto Zoology
Lines: 28
```

and the footer is

```
All work is one man's work.             | Henry Spencer @ U of Toronto Zoology
                    - Kipling           |  henry@zoo.toronto.edu  utzoo!henry
```

Note that this document belongs to the topic **sci.space** (i.e. astronomy).

Recall the text vectorization layer from Problem 1(a). An interesting token that appears in our vocabulary space is the token `henryzootorontoedu`, appearing at index 1121. This token also appears in the example text above.

In [34]:
idx = text_vectorization_glove.get_vocabulary().index('henryzootorontoedu')
print('Token number %d is henryzootorontoedu' % idx)

Token number 1121 is henryzootorontoedu


This token actually appears in quite a few of the training examples. Notice that all of the labels of these documents is 4, corresponding to the category **sci.space**

In [35]:
train_df.loc[np.where(X_train_glove == idx)[0]]

,text,label
211,From: keithley@apple.com (Craig Keithley)\nSub...,4
219,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
271,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
271,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
274,From: sysmgr@king.eng.umd.edu (Doug Mohney)\nS...,4
...,...,...
2874,From: dong@oakhill.sps.mot.com (Don M. Gibson)...,4
2960,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
2960,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
3197,From: Leigh Palmer <palmer@sfu.ca>\nSubject: R...,4


Looking at our GloVe embeddings, the embedding for `henryzootorontoedu` is the all zeros vector.

In [36]:
print('The embedding for token %d is' % idx, embedding_matrix[idx, :])

The embedding for token 1121 is [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Using the information above regarding this token `henryzootorontoedu`, please explain why BoW has an advantage of GloVe (with no fine-tuning). What do you think will happen to the embedding of `henryzootorontoedu` if we do perform fine-tuning?

<font color='red'>**Answer:**</font>

We see that `henryzootorontoedu` is highly correlated with sci-space. BoW model will learn this very quickly. On the other hand GloVe will not be able to detect this pattern as this `henryzootorontoedu` is not a part of GloVe embeddings so it will asign zero vector to it, but with fine tuning it may be able to achieve some sort of similar vector with the existing embeddings in the vocab.

## Part (b): Stripping the Header [10 Points]

It turns out that we can strip the header and footer from the training and testing examples pretty easily. For example, the same document above is now shown below:

In [37]:
newsgroups_train_raw = fetch_20newsgroups(subset='train', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'], remove=('headers', 'footers', 'quotes'))
newsgroups_test_raw = fetch_20newsgroups(subset='test', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'], remove=('headers', 'footers', 'quotes'))

train_df_raw = pd.DataFrame({'text': newsgroups_train_raw.data, 'label': newsgroups_train_raw.target})
test_df_raw = pd.DataFrame({'text': newsgroups_test_raw.data, 'label': newsgroups_test_raw.target})

print(train_df_raw['text'][2101])


The problem with Commercial Titan is that MM has made little or no attempt
to market it.  They're basically happy with their government business and
don't want to have to learn how to sell commercially.

A secondary problem is that it is a bit big.  They'd need to go after
multi-satellite launches, a la Ariane, and that complicates the marketing
task quite significantly.

They also had some problems with launch facilities at just the wrong time
to get them started properly.  If memory serves, the pad used for the Mars
Observer launch had just come out of heavy refurbishment work that had
prevented launches from it for a year or so.

There have been a few CT launches.  Mars Observer was one of them.  So
was that stranded Intelsat, and at least one of its brothers that reached
orbit properly.


Now, copy the code from Problem 1(a), to train a BoW model, but this time adapt the corpus and use `train_df_raw` and `test_df_raw` instead of `train_df` and `test_df`. Note that we name this model `bow_model_raw` instead of `bow_model`.

In [38]:
max_tokens = 2000

text_vectorization = keras.layers.TextVectorization(
    output_mode='multi_hot',max_tokens=max_tokens,standardize='lower_and_strip_punctuation', split='whitespace',ngrams=1
)

text_vectorization.adapt(train_df_raw['text'])

X_train_raw = text_vectorization(train_df_raw['text'])
X_test_raw = text_vectorization(test_df_raw['text'])

### YOUR CODE BELOW ###
inputs = keras.layers.Input(shape=(max_tokens,))

x = keras.layers.Dense(128, activation='relu')(inputs)
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

bow_model_raw = keras.Model(inputs, outputs)
bow_model_raw.summary()

bow_model_raw.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

bow_model_raw.fit(
    x=X_train_raw[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model_raw.evaluate(x=X_test_raw, y=y_test)[1])

bow_model_raw.fit(
    x=X_train_raw, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model_raw.evaluate(x=X_test_raw, y=y_test)[1])

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │       256,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,902 (1003.52 KB)

 Trainable params: 256,902 (1003.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - accuracy: 0.2200 - loss: 1.7559
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7900 - loss: 1.3253 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9000 - loss: 1.0499 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9400 - loss: 0.8269 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9500 - loss: 0.6539 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9700 - loss: 0.5246 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9750 - loss: 0.4284 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.3562 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.3016 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.2592 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.2259 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.1993 
E

Fill in the table below with the test accuracy. Comment on how BoW's performance changes when trained and tested on the 20-newsgroup dataset _without headers and footers_. Does this support our hypothesis from Part (a)?

<font color='red'>**Your Answer Here**</font>

Accuracy Scores
* 200 Training Examples: 49.8.0%
* All Examples: 67.3%

Models performance reduced significantly. This verifies our hypothesis earlier that BoW model uses header for getting better classification results.



## Part (c): Build a Fine-Tuned Model [10 Points]

Now, let's fine tune the GloVe embeddings for our particular dataset. Copy your code from Problem 2, but now change `trainable=True` in `embedding_layer`. Do not change anything else about the model.

In [41]:
embedding_layer = keras.layers.Embedding(
    max_tokens, embedding_dim,embeddings_initializer=keras.initializers.Constant(embedding_matrix), trainable=True, mask_zero=True
)

### YOUR CODE BELOW ###
inputs = keras.Input(shape=(max_length,))
x = embedding_layer(inputs)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

glove_model_tuned = keras.Model(inputs, outputs)
glove_model_tuned.summary()

# Compile model using Adam
glove_model_tuned.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 20 epochs and batch size of 32
glove_model_tuned.fit(
    x=X_train_glove[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples %.4f ***\n" % glove_model_tuned.evaluate(x=X_test_glove, y=y_test)[1])

# Fit model on the training data with 20 epochs and batch size of 32
glove_model_tuned.fit(
    x=X_train_glove, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples %.4f ***\n" % glove_model_tuned.evaluate(x=X_test_glove, y=y_test)[1])

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 400)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 400, 300)  │    600,000 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 400)       │          0 │ input_layer_10[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ embedding_4[0][0… │
│ (GlobalAveragePool… │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 128)       │     38,528 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 128)       │     16,512 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_22[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 6)         │        774 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 655,814 (2.50 MB)

 Trainable params: 655,814 (2.50 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 266ms/step - accuracy: 0.2050 - loss: 1.7736
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2600 - loss: 1.7486 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3000 - loss: 1.7150 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3500 - loss: 1.6784 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4350 - loss: 1.6222 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4350 - loss: 1.5837 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4250 - loss: 1.5124 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5800 - loss: 1.4057 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5950 - loss: 1.3047 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6750 - loss: 1.2025 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7500 - loss: 1.0646 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7700 - loss: 0.9519 


Report the performance of our Glove Model above with fine tuning. Comment on this performance and compare it to that of the GloVe model without fine tuning. Explain when fine-tuning can be helpful and/or hurtful.

**<font color='red'>Answere:</font>**


* 200 Examples: 69.0%
* All Examples: 80.0%

Fine tuning made the performance better for all examples and not that good for 200 examples. This makes sense, when we have only a few examples, fine tuning leads to a lot of overfitting. On the other hand, when we have many examples, fine tuning leads to better embedding that are tailored towasrds a particular classification task that cap and thus better overall performance.

The embedding for the token `henryzootorontoedu` is now no longer all 0's. Nice!

In [42]:
print('The embedding for token %d is' % idx, glove_model_tuned.layers[1].weights[0][idx, :])

The embedding for token 1121 is tf.Tensor(
[-0.08904565 -0.21201457  0.0388522  -0.1125812  -0.18976809 -0.11572155
  0.02549486  0.01980693 -0.03588753 -0.15655196  0.23540042 -0.01740871
  0.13290575  0.12085324 -0.17550048  0.16100806  0.18953094  0.20765379
  0.14312637  0.01448543  0.17707759 -0.13048773  0.18191263  0.20617145
  0.17977582  0.15172979  0.11045417  0.13378835  0.09093418 -0.09876261
  0.20621744 -0.18492137  0.09659029  0.08153679 -0.14938107 -0.09716842
  0.04238201  0.12263861  0.1728274   0.13193008 -0.10281297  0.12627245
 -0.13714506  0.11211809 -0.19778928 -0.14486915 -0.09665868  0.17900187
  0.01889497 -0.17651246 -0.15251604  0.08329385 -0.21031243  0.22220862
 -0.1226134   0.17252265  0.07931257  0.19300574  0.01005186 -0.11010591
 -0.15498286  0.01707778  0.06101321 -0.15910204  0.06341874 -0.0457495
 -0.02321424  0.19782333  0.02475138 -0.0875887   0.20303354  0.16032724
  0.13685837  0.14135416  0.00211947  0.12636949  0.07568571  0.02007418
  0.15953

# Problem 4: BERT Transformer Model [15 Points]

Finally, we use will a famous transformer pre-trained model called [Bert](https://en.wikipedia.org/wiki/BERT_(language_model)). Just like how ResNet was a pre-trained model for computer vision (image processing), BERT is a pre-trained model for natural language processing.

In [6]:
!pip install -q -U tensorflow-text ## install the package for NLP tasks in tensorflow

In [7]:
import tensorflow_hub as hub
import tensorflow_text

bert_preprocess = 'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3'

bert_layers = 12
bert_units = 768
bert_heads = 12

bert_encoder = f'https://tfhub.dev/tensorflow/bert_en_uncased_L-{bert_layers}_H-{bert_units}_A-{bert_heads}/4'

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Let's look to some examples of the text processing required for BERT:

In [8]:
bert_preprocess_model = hub.KerasLayer(bert_preprocess)

text_test = ['This is comedy as it once was and comparing this with the two remakes.']
text_preprocessed = bert_preprocess_model(text_test)
print(f'Word Ids   : {text_preprocessed["input_word_ids"]}')

text_test = ['Although I rated this movie a 2 for showing a complete lack of effort in trying to create a quality horror film it was a 10 on the unintentional funny scale.']
text_preprocessed = bert_preprocess_model(text_test)
print(f'Word Ids   : {text_preprocessed["input_word_ids"]}')

Word Ids   : [[  101  2023  2003  4038  2004  2009  2320  2001  1998 13599  2023  2007
   1996  2048 12661  2015  1012   102     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]]
Word Ids   : [[  101  2348  1045  6758  2023  3185  1037  1016  2005  4760  1037  3143
   3768  1997  3947  1999  2667  2000  3443  1037  3737  5469  2143  2009
   2001  1037  2184  2006  1996  4

Notice how all examples start with the token `101` and end with `102` (and then followed by `[PAD]` tokens as indicated by 0). Those are special placeholder tokens that can be used for different purposes. For this exercise, we will only concern about the first one, and we will call this token the 'classification token', or `[CLS]`.

The two functions below are used to convert text into tokens (`bert_textvect`) and tokens into embeddings (`bert_features`). You do not need to understand exactly how the functions below work. Note that the following cell may take some time to run (up to 5 minutes).

In [ ]:
max_length = 512

# Load the TensorFlow Hub models directly
preprocessor = hub.load(bert_preprocess)
encoder = hub.load(bert_encoder)
def bert_textvect(x):
    """
    Converts a list of strings into BERT input tokens.
    """
    texts = tf.constant(x.astype(str).tolist(), dtype=tf.string)

    # Tokenize the text
    tokenized_input = preprocessor.tokenize(texts)

    # Pack the tokens into BERT input format
    output = preprocessor.bert_pack_inputs(
        [tokenized_input],
        seq_length=max_length
    )

    return output

def bert_features(x, batch_size=16):
    """
    Converts BERT tokens into 768-dimensional embeddings.
    Uses the [CLS] token representation.
    """
    all_features = []

    n = int(x["input_word_ids"].shape[0])

    for start in range(0, n, batch_size):
        # Select a batch
        batch = {
            key: value[start:start + batch_size]
            for key, value in x.items()
        }

        # Extract BERT features
        output = encoder(batch)

        # Take the [CLS] token embedding
        features = output["sequence_output"][:, 0, :].numpy()
        all_features.append(features)

    return np.concatenate(all_features, axis=0)

X_bert_train = bert_textvect(train_df["text"])
X_bert_test = bert_textvect(test_df["text"])

features_train = bert_features(X_bert_train)
features_test = bert_features(X_bert_test)

print(features_train.shape)
print(features_test.shape)

Next, we'll use the BERT output embedding for the CLS token as input to train a simple neural net. Please build a neural network with 2 dense layers of 128 neurons each using a ReLU activation. Each dense layer should be followed by a Dropout layer with probability of 0.1.

In [ ]:
### YOUR CODE BELOW ###
input = keras.Input(shape=(bert_units, ))

x = keras.layers.Dense(128, activation='relu')(input)
x = keras.layers.Dropout(0.1)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.1)(x)

output = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)
### YOUR CODE ABOVE ###

# Model
bert_model = keras.Model(input, output)
bert_model.summary()

bert_model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

bert_model.fit(
    x=features_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with 200 Examples %.4f ***\n" % bert_model.evaluate(x=features_test, y=y_test)[1])

bert_model.fit(
    x=features_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)
print("\n*** Test accuracy with All Examples %.4f ***\n" % bert_model.evaluate(x=features_test, y=y_test)[1])

Fill in the table below with the accuracies from above. Comment on the performance of the model relative to the BoW and GloVe models.

<font color='red'>**Your Answer Here**</font>

BERT Model Accuracy
* 200 Examples: 0.0%
* All Examples: 0.0%

_Replace this with your answer_

# Conclusion [5 Points]

Please discuss any general take-aways from this homework exercise.

<font color='red'>**Your Answer Here**</font>

_Replace this with your answer_